In [ ]:
# ==========================================
# 1. Import Libraries
# ==========================================
import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam

# ==========================================
# 2. Load Dataset
# ==========================================
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

x_train = x_train / 255.0
x_test = x_test / 255.0

y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

# ==========================================
# 3. Create Data Pipeline (IMPORTANT FIX)
# ==========================================
BATCH_SIZE = 32

def preprocess(image, label):
    image = tf.image.resize(image, (224, 224))  # resize per batch
    return image, label

train_ds = tf.data.Dataset.from_tensor_slices((x_train, y_train))
train_ds = train_ds.map(preprocess).batch(BATCH_SIZE).prefetch(1)

test_ds = tf.data.Dataset.from_tensor_slices((x_test, y_test))
test_ds = test_ds.map(preprocess).batch(BATCH_SIZE).prefetch(1)

# ==========================================
# 4. Load VGG16
# ==========================================
base_model = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze layers
for layer in base_model.layers:
    layer.trainable = False

# ==========================================
# 5. Build Model
# ==========================================
model = models.Sequential([
    base_model,
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
])

# ==========================================
# 6. Compile
# ==========================================
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# ==========================================
# 7. Train (Frozen)
# ==========================================
print("Training with frozen layers...")
model.fit(train_ds, epochs=5)

# ==========================================
# 8. Fine-tuning
# ==========================================
for layer in base_model.layers[-4:]:
    layer.trainable = True

model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Fine-tuning...")
model.fit(train_ds, epochs=5)

# ==========================================
# 9. Evaluate
# ==========================================
loss, acc = model.evaluate(test_ds)
print(f"Accuracy: {acc*100:.2f}%")

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Training with frozen layers...
Epoch 1/5
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 303s 184ms/step - accuracy: 0.0987 - loss: 2.3112
Epoch 2/5
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 285s 182ms/step - accuracy: 0.0959 - loss: 2.3028
Epoch 3/5
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 285s 182ms/step - accuracy: 0.0959 - loss: 2.3030
Epoch 4/5
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 285s 182ms/step - accuracy: 0.0965 - loss: 2.3028
Epoch 5/5
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 285s 182ms/step - accuracy: 0.0965 - loss: 2.3028
Fine-tuning...
Epoch 1/5
1189/1563 ━━━━━━━━━━━━━━━━━━━━ 1:17 208ms/step - accuracy: 0.0960 - loss: 2.3028

In [1]:
# ==========================================
# 1. Import Libraries
# ==========================================
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam

# ==========================================
# 2. Load Dataset
# ==========================================
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

# Normalize
x_train = x_train / 255.0
x_test = x_test / 255.0

# One-hot encoding
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

# ==========================================
# 3. Resize ONCE (IMPORTANT FIX)
# ==========================================
IMG_SIZE = 96

x_train = tf.image.resize(x_train, (IMG_SIZE, IMG_SIZE))
x_test = tf.image.resize(x_test, (IMG_SIZE, IMG_SIZE))

# ==========================================
# 4. Create Dataset Pipeline
# ==========================================
BATCH_SIZE = 64

train_ds = tf.data.Dataset.from_tensor_slices((x_train, y_train)) \
    .shuffle(10000) \
    .batch(BATCH_SIZE) \
    .prefetch(tf.data.AUTOTUNE)

test_ds = tf.data.Dataset.from_tensor_slices((x_test, y_test)) \
    .batch(BATCH_SIZE) \
    .prefetch(tf.data.AUTOTUNE)

# ==========================================
# 5. Load MobileNetV2
# ==========================================
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

# Freeze base model
for layer in base_model.layers:
    layer.trainable = False

# ==========================================
# 6. Build Model
# ==========================================
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
])

# ==========================================
# 7. Compile
# ==========================================
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# ==========================================
# 8. Train (Frozen Layers)
# ==========================================
print("Training with frozen layers...")
model.fit(train_ds, epochs=5, validation_data=test_ds)

# ==========================================
# 9. Fine-Tuning (UNFREEZE LAST LAYERS)
# ==========================================
for layer in base_model.layers[-20:]:
    layer.trainable = True

model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Fine-tuning...")
model.fit(train_ds, epochs=5, validation_data=test_ds)

# ==========================================
# 10. Evaluate
# ==========================================
loss, acc = model.evaluate(test_ds)
print(f"Final Accuracy: {acc*100:.2f}%")

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Training with frozen layers...
Epoch 1/5


: 

: 

: 